In [15]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import parselmouth
from dtw import dtw
from numpy.linalg import norm
from resemblyzer import VoiceEncoder, preprocess_wav

In [16]:
# === 1. Load WAVs ===
source_path = "/root/music_wav/Lenka - The Show.wav"
converted_path = "/root/The_show.wav"
sr = 44000
source, _ = librosa.load(source_path, sr=sr)
converted, _ = librosa.load(converted_path, sr=sr)

In [17]:
# === 2. Resemblyzer Speaker Embedding Shift ===
encoder = VoiceEncoder()
embed_source = encoder.embed_utterance(preprocess_wav(source_path))
embed_converted = encoder.embed_utterance(preprocess_wav(converted_path))

cosine_sim = np.dot(embed_source, embed_converted) / (np.linalg.norm(embed_source) * np.linalg.norm(embed_converted))
print(f"Cosine Similarity (source vs converted): {cosine_sim:.4f}")

Loaded the voice encoder model on cuda in 0.01 seconds.
Cosine Similarity (source vs converted): 0.8941


In [18]:
from dtw import dtw
from scipy.spatial.distance import cdist

def extract_mfcc(wav, sr, n_mfcc=13):
    mfcc = librosa.feature.mfcc(y=wav, sr=sr, n_mfcc=n_mfcc)
    return mfcc.T  # shape: (T, D)

mfcc_source = extract_mfcc(source, sr)
mfcc_converted = extract_mfcc(converted, sr)

# Compute pairwise distance matrix manually
dist_matrix = cdist(mfcc_source, mfcc_converted, metric='euclidean')

# Call DTW without custom distance function
alignment = dtw(mfcc_source, mfcc_converted, keep_internals=True)
mcd = alignment.normalizedDistance
print(f"📉 MCD (fallback with default distance): {mcd:.4f}")

📉 MCD (fallback with default distance): 72.5096


In [19]:
# === 4. F0 Consistency ===
def extract_f0(wav_path):
    snd = parselmouth.Sound(wav_path)
    pitch = snd.to_pitch()
    f0 = pitch.selected_array['frequency']
    return f0[f0 > 0]

f0_source = extract_f0(source_path)
f0_converted = extract_f0(converted_path)
min_len = min(len(f0_source), len(f0_converted))
f0_source = f0_source[:min_len]
f0_converted = f0_converted[:min_len]

rmse = np.sqrt(np.mean((f0_source - f0_converted)**2))
corr = np.corrcoef(f0_source, f0_converted)[0, 1]
print(f"F0 RMSE: {rmse:.2f}")
print(f"F0 Correlation: {corr:.4f}")

F0 RMSE: 96.41
F0 Correlation: 0.2525
